# Simple Moving-Average Crossover

This long/cash strategy uses a 31-session and a 63-session simple moving average of adjusted closing prices. After the initial 31-session warm-up, the signal is long when the fast average is above the slow average. Each signal sets a fixed position of 100 shares or cash at the next session's adjusted close.

The recovered example used FEYE. It now uses IBM, another stock from the original notebook's universe, and preserves the original 2017–2019 dates. The backtest uses a configurable starting balance and omits trading costs. [FEYE ticker change](https://www.sec.gov/Archives/edgar/data/1370880/000137088021000039/a8-kfor10x4x2021xex991.htm)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from _market_data import download_ohlcv


SYMBOL = "IBM"
START_DATE = "2017-12-12"
END_DATE = "2019-04-09"
FAST_SESSIONS = 31
SLOW_SESSIONS = 63
POSITION_SHARES = 100
INITIAL_CASH = 25_000.0


close = download_ohlcv(SYMBOL, START_DATE, END_DATE, auto_adjust=True)["Close"]
results = pd.DataFrame({"close": close})
results["fast_sma"] = close.rolling(FAST_SESSIONS).mean()
results["slow_sma"] = close.rolling(SLOW_SESSIONS, min_periods=1).mean()
long_signal = results["fast_sma"].gt(results["slow_sma"])
results["shares"] = long_signal.shift(1, fill_value=False).astype(int) * POSITION_SHARES
results["trade_shares"] = results["shares"].diff().fillna(results["shares"])
results["cash"] = INITIAL_CASH - (results["trade_shares"] * close).cumsum()
results["holdings"] = results["shares"] * close
results["equity"] = results["cash"] + results["holdings"]
results["drawdown"] = results["equity"] / results["equity"].cummax() - 1.0

daily_returns = results["equity"].pct_change().dropna()
daily_volatility = daily_returns.std()
sharpe = np.sqrt(252) * daily_returns.mean() / daily_volatility if daily_volatility > 0 else np.nan
print(f"Sessions: {len(results)}")
print(f"Orders: {(results['trade_shares'] != 0).sum()}")
print(f"Annualized Sharpe ratio with zero risk-free rate: {sharpe:.2f}")

fig, axes = plt.subplots(3, 1, sharex=True, figsize=(12, 9))
results[["close", "fast_sma", "slow_sma"]].plot(ax=axes[0], title=f"{SYMBOL} Adjusted Prices")
buys = results.loc[results["trade_shares"].gt(0), "close"]
sells = results.loc[results["trade_shares"].lt(0), "close"]
axes[0].scatter(buys.index, buys, marker="^", color="green", label="Buy")
axes[0].scatter(sells.index, sells, marker="v", color="red", label="Sell")
axes[0].legend()
results["equity"].plot(ax=axes[1], title="Portfolio Value")
results["drawdown"].plot(ax=axes[2], title="Drawdown")
fig.tight_layout()
plt.show()
